In [ ]:
pip install pygame

In [ ]:
import os  
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"  # 7al moshkelet el MKL libraries lama tet3ared
import pygame  
import sys  
import numpy as np  
import time  
import copy  
import random  
import math  

In [ ]:
pygame.init()

# Asasy board size 
DEFAULT_BOARD_SIZE = 19
MAX_BOARD_SIZE = 19 

CELL_SIZE = 25
STONE_RADIUS = 12  
MARGIN = 30  
BUTTON_HEIGHT = 30  
BUTTON_MARGIN = 8
INFO_HEIGHT = 110
SETTINGS_HEIGHT = 130

# Raise GUI elements
VERTICAL_PADDING = 28  # Ytzawed to raise 
HORIZONTAL_PADDING = 0 # Placeholder, will be calculated dynamically

BLACK = (0, 0, 0)
WHITE = (255, 255, 255)
BOARD_COLOR = (220, 179, 92)
GRID_COLOR = (0, 0, 0)
RED = (255, 0, 0)
GOLD = (255, 215, 32)
BLUE = (100, 149, 237)
LIGHT_BLUE = (173, 216, 230)
GREEN = (50, 205, 50)
LIGHT_GREEN = (144, 238, 144)
GRAY = (200, 200, 200)
DARK_GRAY = (100, 100, 100)
BEIGE = (245, 245, 220)

WINDOW_WIDTH = MARGIN * 2 + CELL_SIZE * MAX_BOARD_SIZE
WINDOW_HEIGHT = MARGIN * 2 + CELL_SIZE * MAX_BOARD_SIZE + BUTTON_HEIGHT + BUTTON_MARGIN + INFO_HEIGHT + SETTINGS_HEIGHT


# Ebda2 screen with fixed size
screen = pygame.display.set_mode((0, 0), pygame.FULLSCREEN) # Modified for fullscreen
pygame.display.set_caption("Connect-6 Game")

# --- Calculate Horizontal Padding for Fullscreen Centering ---
infoObject = pygame.display.Info()
FULLSCREEN_WIDTH = infoObject.current_w
HORIZONTAL_PADDING = (FULLSCREEN_WIDTH - WINDOW_WIDTH) // 2
HORIZONTAL_PADDING = max(0, HORIZONTAL_PADDING) # Ensure padding is not negative
# --- End Horizontal Padding Calculation ---

font = pygame.font.SysFont("Arial", 12)
large_font = pygame.font.SysFont("Arial", 16)
small_font = pygame.font.SysFont("Arial", 11)
Big_font = pygame.font.SysFont("Arial", 48)

class Connect6Board:
    def __init__(self, size=DEFAULT_BOARD_SIZE):
        self.size = size
        self.grid = np.zeros((size, size), dtype=int)
        self.move_history = []
        self.current_player = 1  # 1 for Black, 2 for White
        self.first_move = True   # Tabe3 first move (Black places only one stone)
        self.game_over = False
        self.winner = None
        self.last_move = None
        self.winning_stones = []  # Store winning stone positions
        self.stones_placed_this_turn = 0
        self.turn_count = 0  # Tabe3 total turns

    def reset(self):
        self.grid = np.zeros((self.size, self.size), dtype=int)
        self.move_history = []
        self.current_player = 1
        self.first_move = True
        self.game_over = False
        self.winner = None
        self.last_move = None
        self.winning_stones = []
        self.stones_placed_this_turn = 0
        self.turn_count = 0

    def is_valid_move(self, row, col):
        if not (0 <= row < self.size and 0 <= col < self.size):
            return False

        # Make sure if position is empty
        return self.grid[row, col] == 0

    def place_stone(self, row, col):
        if not self.is_valid_move(row, col) or self.game_over:
            return False

        self.grid[row, col] = self.current_player
        self.move_history.append((row, col, self.current_player))
        self.last_move = (row, col)
        self.stones_placed_this_turn += 1

        # Make sure for win
        if self.check_win(row, col):
            self.game_over = True
            self.winner = self.current_player
            return True

        # Make sure if turn is complete
        stones_per_turn = 1 if self.first_move else 2
        if self.stones_placed_this_turn >= stones_per_turn:
            self.next_turn()

        return True

    def check_win(self, row, col):
        player = self.grid[row, col]
        directions = [
            (1, 0),   
            (0, 1),
            (1, 1),   
            (1, -1)   
        ]

        for dr, dc in directions:
            count = 1  # Count the stone 
            stones = [(row, col)]

            r, c = row + dr, col + dc
            while 0 <= r < self.size and 0 <= c < self.size and self.grid[r, c] == player:
                count += 1
                stones.append((r, c))
                r += dr
                c += dc

            r, c = row - dr, col - dc
            while 0 <= r < self.size and 0 <= c < self.size and self.grid[r, c] == player:
                count += 1
                stones.append((r, c))
                r -= dr
                c -= dc

            # Make sure if there are 6 or more stones in a row
            if count >= 6:
                self.winning_stones = stones
                return True

        if np.count_nonzero(self.grid) == self.size * self.size:
            self.game_over = True
            return False

        return False

    def next_turn(self):
        if self.first_move:
            self.first_move = False
            self.current_player = 3 - self.current_player
        else:
            self.current_player = 3 - self.current_player  # Switch between 1 and 2

        self.stones_placed_this_turn = 0
        self.turn_count += 1

    def get_available_moves(self):
        return [(r, c) for r in range(self.size) for c in range(self.size) if self.grid[r, c] == 0]

    def get_player_name(self, player=None):
        if player is None:
            player = self.current_player
        return "Black" if player == 1 else "White"

    def get_state_string(self):
        if self.game_over:
            if self.winner:
                return f"Game Over: {self.get_player_name(self.winner)} wins!"
            else:
                return "Game Over: Draw!"
        else:
            player_name = self.get_player_name()
            stones_left = 1 if self.first_move and self.stones_placed_this_turn == 0 else 2 - self.stones_placed_this_turn
            stones = f"{stones_left} stone{"s" if stones_left > 1 else ""}"
            return f"{player_name}\'s turn to place {stones}"


    def undo_last_move(self):
        if not self.move_history:
            return False

        # Geeb el last move
        row, col, player = self.move_history.pop()

        self.grid[row, col] = 0

        # Reset game over state if needed
        self.game_over = False
        self.winner = None
        self.winning_stones = []

        # Updating current player and turn state
        if self.stones_placed_this_turn > 0:
            self.stones_placed_this_turn -= 1
        else:
            if self.first_move:
                pass
            else:
                self.current_player = 3 - self.current_player
                if len(self.move_history) <= 1:
                    self.first_move = True
                    self.stones_placed_this_turn = 0
                else:
                    self.stones_placed_this_turn = 1

        if self.move_history:
            self.last_move = (self.move_history[-1][0], self.move_history[-1][1])
        else:
            self.last_move = None

        return True

    def get_board_state_hash(self):
        return hash(self.grid.tobytes())

class Connect6Heuristics:

    @staticmethod
    def basic_pattern_heuristic(board, player_number):
        score = 0
        opponent_number = 3 - player_number

        directions = [
            (1, 0),   
            (0, 1),
            (1, 1),
            (1, -1)   
        ]

        pattern_scores = {
            1: 1,      
            2: 10,     
            3: 100,
            4: 1000, 
            5: 10000,  
            6: 100000  # 6 consecutive stones (win)
        }

        # Make sure patterns for both players
        for player in [player_number, opponent_number]:
            player_score = 0

           
            for direction in directions:
                dr, dc = direction

                
                for r in range(board.size):
                    for c in range(board.size):
                        if (r + 5*dr >= board.size or r + 5*dr < 0 or
                            c + 5*dc >= board.size or c + 5*dc < 0):
                            continue

                       
                        consecutive = 0
                        empty_ends = 0

                        if (r - dr >= 0 and r - dr < board.size and
                            c - dc >= 0 and c - dc < board.size and
                            board.grid[r - dr, c - dc] == 0):
                            empty_ends += 1

                        if (r + 6*dr >= 0 and r + 6*dr < board.size and
                            c + 6*dc >= 0 and c + 6*dc < board.size and
                            board.grid[r + 6*dr, c + 6*dc] == 0):
                            empty_ends += 1

                        
                        for i in range(6):
                            nr, nc = r + i*dr, c + i*dc
                            if board.grid[nr, nc] == player:
                                consecutive += 1
                            elif board.grid[nr, nc] != 0:  # Opponent"s stone
                                consecutive = 0
                                break

                        if consecutive > 0:
                            
                            openness_bonus = 1.0
                            if empty_ends == 1:
                                openness_bonus = 1.5
                            elif empty_ends == 2:
                                openness_bonus = 2.0  

                            pattern_value = pattern_scores.get(consecutive, 0) * openness_bonus
                            player_score += pattern_value

            if player == player_number:
                score += player_score
            else:
                score -= player_score * 1.2  # prioritize defense

        return score

    @staticmethod
    def _evaluate_critical_patterns(board, player_number):
        """Evaluates immediate winning threats (5-in-a-row) and strong threats (open 4-in-a-row)."""
        score = 0
        opponent_number = 3 - player_number
        directions = [(1, 0), (0, 1), (1, 1), (1, -1)]

        WIN_SCORE = 50000  # Score for immediate win/block threat (5-in-a-row)
        STRONG_THREAT_SCORE = 5000 # Score for open 4-in-a-row

        for player in [player_number, opponent_number]:
            player_threat_score = 0
            for r in range(board.size):
                for c in range(board.size):
                    if board.grid[r, c] == player:
                        for dr, dc in directions:
                            # Check for 5-in-a-row with at least one open end
                            consecutive = 1
                            open_ends = 0
                            line_stones = [(r, c)]

                            # Check forward
                            for i in range(1, 6):
                                nr, nc = r + i * dr, c + i * dc
                                if 0 <= nr < board.size and 0 <= nc < board.size:
                                    if board.grid[nr, nc] == player:
                                        consecutive += 1
                                        line_stones.append((nr, nc))
                                    elif board.grid[nr, nc] == 0:
                                        open_ends += 1
                                        break # Stop counting in this direction
                                    else: # Opponent stone
                                        break
                                else:
                                    break # Out of bounds

                            # Check backward
                            for i in range(1, 6):
                                nr, nc = r - i * dr, c - i * dc
                                if 0 <= nr < board.size and 0 <= nc < board.size:
                                    if board.grid[nr, nc] == player:
                                        consecutive += 1
                                        line_stones.append((nr, nc))
                                    elif board.grid[nr, nc] == 0:
                                        open_ends += 1
                                        break # Stop counting in this direction
                                    else: # Opponent stone
                                        break
                                else:
                                    break # Out of bounds

                            # Score based on findings for this specific line starting at (r,c)
                            # Ensure we only count each line once by checking if the first stone in the line is (r,c)
                            if consecutive >= 5 and open_ends >= 1:
                                # Check if (r,c) is the start of this line segment to avoid double counting
                                nr_before, nc_before = r - dr, c - dc
                                if not (0 <= nr_before < board.size and 0 <= nc_before < board.size and board.grid[nr_before, nc_before] == player):
                                    player_threat_score += WIN_SCORE

                            elif consecutive == 4 and open_ends == 2:
                                # Check if (r,c) is the start of this line segment
                                nr_before, nc_before = r - dr, c - dc
                                if not (0 <= nr_before < board.size and 0 <= nc_before < board.size and board.grid[nr_before, nc_before] == player):
                                    player_threat_score += STRONG_THREAT_SCORE

            if player == player_number:
                score += player_threat_score
            else:
                score -= player_threat_score * 1.5 # Prioritize blocking opponent threats more

        return score

    @staticmethod
    def advanced_weighted_heuristic(board, player_number):
        # Start with basic heuristic score
        score = Connect6Heuristics.basic_pattern_heuristic(board, player_number)

        # Add score for critical threats (win/block 5-in-a-row, open 4-in-a-row)
        critical_score = Connect6Heuristics._evaluate_critical_patterns(board, player_number)
        score += critical_score

        opponent_number = 3 - player_number

        center = board.size // 2
        center_region = min(3, board.size // 6)

       
        center_bonus = 0
        for r in range(center - center_region, center + center_region + 1):
            for c in range(center - center_region, center + center_region + 1):
                if 0 <= r < board.size and 0 <= c < board.size:
                    if board.grid[r, c] == player_number:
                        distance = max(abs(r - center), abs(c - center))
                        center_bonus += (center_region + 1 - distance) * 5
                    elif board.grid[r, c] == opponent_number:
                        distance = max(abs(r - center), abs(c - center))
                        center_bonus -= (center_region + 1 - distance) * 5

        score += center_bonus

       
        player_groups = Connect6Heuristics._count_connected_groups(board, player_number)
        opponent_groups = Connect6Heuristics._count_connected_groups(board, opponent_number)

        score += player_groups * 50
        score -= opponent_groups * 60

        return score

    @staticmethod
    def _count_connected_groups(board, player_number):
        marked = np.zeros((board.size, board.size), dtype=bool)
        group_count = 0

        directions = [(0, 1), (1, 0), (0, -1), (-1, 0), (1, 1), (-1, -1), (1, -1), (-1, 1)]

        # Depth-first search
        def dfs(r, c):
            if (not (0 <= r < board.size and 0 <= c < board.size) or
                marked[r, c] or board.grid[r, c] != player_number):
                return 0

            marked[r, c] = True
            size = 1

            for dr, dc in directions:
                size += dfs(r + dr, c + dc)

            return size

        for r in range(board.size):
            for c in range(board.size):
                if board.grid[r, c] == player_number and not marked[r, c]:
                    group_size = dfs(r, c)
                    if group_size >= 2:  
                        group_count += 1

        return group_count

class Connect6AI:
    def __init__(self, player_number, heuristic_type="basic"):
        self.player_number = player_number
        self.opponent_number = 3 - player_number
        self.heuristic_type = heuristic_type

        # Set search parameters based on heuristic type
        if self.heuristic_type == "advanced":
            self.search_depth = 3  # Increased depth for advanced AI
            self.max_time = 6.0    # Increased time for advanced AI
        else:
            self.search_depth = 2  # Default for basic AI
            self.max_time = 3.0    # Default for basic AI

        # Transposition table for storing evaluated positions
        self.transposition_table = {}

        self.nodes_evaluated = 0
        self.start_time = 0

    def get_move(self, board):
        self.start_time = time.time()
        self.nodes_evaluated = 0

        # If this is the first move of the game, play in the center
        if board.first_move and len(board.move_history) == 0:
            return (board.size // 2, board.size // 2)

        # Geeb available moves
        available_moves = self._get_smart_moves(board)

        if board.first_move and board.stones_placed_this_turn == 0:
            # Choose the best single move
            best_move = self._get_best_single_move(board, available_moves)
            return best_move

        # For two stones, get the best 2 moves
        best_moves = self._get_best_move_pair(board, available_moves)

        return best_moves[0]

    def get_second_move(self, board):
       
        available_moves = self._get_smart_moves(board)

        
        best_moves = self._get_best_move_pair(board, available_moves)

        return best_moves[1] if len(best_moves) > 1 else best_moves[0]

    def _get_best_single_move(self, board, available_moves):
        best_score = float("-inf")
        best_move = None

        # Prioritize immediate winning move
        for move in available_moves:
            row, col = move
            board_copy = copy.deepcopy(board)
            board_copy.place_stone(row, col)
            if board_copy.game_over and board_copy.winner == self.player_number:
                # print(f"AI found immediate single win: {move}") # Debug print
                return move # Found an immediate win, take it!

        # If no immediate win, proceed with minimax evaluation
        current_search_depth = self.search_depth if self.heuristic_type == "advanced" else 2 # Use correct depth
        for move in available_moves:
            row, col = move
            board_copy = copy.deepcopy(board)
            board_copy.place_stone(row, col)
            # No need to check win again here, already done above
            board_copy.next_turn()

            # Evaluate the board state using minimax
            score = self._minimax(board_copy, current_search_depth, False, float("-inf"), float("inf"))

            if score > best_score:
                best_score = score
                best_move = move

            # Make sure if time limit is reached
            if time.time() - self.start_time > self.max_time * 0.9:
                # print("AI time limit reached in single move search") # Debug print
                break

        # print(f"AI best single move (minimax): {best_move} with score {best_score}") # Debug print
        return best_move if best_move else available_moves[0]

    def _get_best_move_pair(self, board, available_moves):
        best_score = float("-inf")
        best_moves = []

        if not available_moves:
            return [] # Should not happen if game is not over
        if len(available_moves) == 1:
            return [available_moves[0], available_moves[0]] # Only one move possible

        # --- Immediate Win/Block Check ---
        # Check if any single move wins immediately (should be caught by get_move, but double check)
        for move1 in available_moves:
            board_copy_win_check = copy.deepcopy(board)
            board_copy_win_check.place_stone(move1[0], move1[1])
            if board_copy_win_check.game_over and board_copy_win_check.winner == self.player_number:
                # Found a win with the first stone, find any valid second stone
                second_move = next((m for m in available_moves if m != move1), available_moves[0] if available_moves[0] != move1 else available_moves[1])
                # print(f"AI found immediate win with first stone: {move1}, returning pair {[move1, second_move]}") # Debug
                return [move1, second_move]

        # Check if opponent has an immediate win (5-in-a-row) and prioritize blocking it
        opponent_winning_moves = self._find_immediate_threats(board, self.opponent_number)
        blocking_pairs = []
        if opponent_winning_moves:
            # print(f"Opponent threats found: {opponent_winning_moves}") # Debug
            must_block_move = opponent_winning_moves[0] # Prioritize blocking the first found threat
            # Check pairs where the first stone blocks the threat
            if must_block_move in available_moves:
                for move2 in available_moves:
                    if move2 != must_block_move:
                        blocking_pairs.append([must_block_move, move2])
            # Check pairs where the second stone blocks the threat
            for move1 in available_moves:
                if move1 != must_block_move and must_block_move in self._get_remaining_moves(board, move1):
                     blocking_pairs.append([move1, must_block_move])

            if blocking_pairs:
                 # print(f"Prioritizing blocking pairs: {blocking_pairs}") # Debug
                 # Evaluate only the blocking pairs using minimax
                 current_search_depth = self.search_depth if self.heuristic_type == "advanced" else 1 # Use correct depth, maybe shallower for blocking
                 best_blocking_score = float("-inf")
                 best_blocking_pair = []
                 for pair in blocking_pairs:
                     move1, move2 = pair
                     board_copy = copy.deepcopy(board)
                     board_copy.place_stone(move1[0], move1[1])
                     # Check win after first stone (unlikely if blocking, but check)
                     if board_copy.game_over and board_copy.winner == self.player_number:
                         return pair
                     board_copy.place_stone(move2[0], move2[1])
                     if board_copy.game_over and board_copy.winner == self.player_number:
                         return pair # Win after placing second stone

                     board_copy.next_turn()
                     score = self._minimax(board_copy, current_search_depth -1, False, float("-inf"), float("inf"))

                     if score > best_blocking_score:
                         best_blocking_score = score
                         best_blocking_pair = pair

                     if time.time() - self.start_time > self.max_time * 0.9:
                         break
                 if best_blocking_pair: # Return the best blocking pair found
                     # print(f"AI chose best blocking pair: {best_blocking_pair} score {best_blocking_score}") # Debug
                     return best_blocking_pair
                 # Fall through if time runs out or no blocking pair is good

        # --- End Immediate Win/Block Check ---

        # Proceed with normal minimax evaluation for pairs
        current_search_depth = self.search_depth if self.heuristic_type == "advanced" else 2 # Use correct depth
        max_moves_to_consider = min(15, len(available_moves)) # Consider top N moves
        considered_moves = available_moves[:max_moves_to_consider]

        # Check pairs for winning moves first
        for i in range(len(considered_moves)):
            for j in range(len(considered_moves)):
                if i == j: continue
                move1 = considered_moves[i]
                move2 = considered_moves[j]

                board_copy = copy.deepcopy(board)
                board_copy.place_stone(move1[0], move1[1])
                if board_copy.game_over and board_copy.winner == self.player_number:
                    # print(f"AI found win with first stone {move1} in pair check") # Debug
                    return [move1, move2]
                # Check if the second move is still valid after the first
                if board_copy.is_valid_move(move2[0], move2[1]):
                    board_copy.place_stone(move2[0], move2[1])
                    if board_copy.game_over and board_copy.winner == self.player_number:
                        # print(f"AI found win with second stone {move2} in pair check") # Debug
                        return [move1, move2]

        # If no immediate win found in pairs, run minimax
        best_score = float("-inf")
        best_moves = []
        evaluated_pairs = set()

        for i in range(len(considered_moves)):
            for j in range(len(considered_moves)):
                if i == j: continue
                move1 = considered_moves[i]
                move2 = considered_moves[j]
                pair_tuple = tuple(sorted((move1, move2)))
                if pair_tuple in evaluated_pairs: continue # Avoid evaluating the same pair twice

                board_copy = copy.deepcopy(board)
                board_copy.place_stone(move1[0], move1[1])
                # Ensure second move is valid before placing
                if not board_copy.is_valid_move(move2[0], move2[1]):
                    continue # Skip this pair if second move becomes invalid
                board_copy.place_stone(move2[0], move2[1])

                # We already checked for immediate wins above
                board_copy.next_turn()
                score = self._minimax(board_copy, current_search_depth - 1, False, float("-inf"), float("inf"))
                evaluated_pairs.add(pair_tuple)

                if score > best_score:
                    best_score = score
                    best_moves = [move1, move2]

                if time.time() - self.start_time > self.max_time * 0.9:
                    # print("AI time limit reached in pair move search") # Debug
                    break
            if time.time() - self.start_time > self.max_time * 0.9:
                break

        # print(f"AI best pair (minimax): {best_moves} with score {best_score}") # Debug
        # Fallback if no moves evaluated or time ran out early
        if not best_moves:
            if available_moves:
                move1 = available_moves[0]
                move2 = available_moves[1] if len(available_moves) > 1 else available_moves[0]
                best_moves = [move1, move2]
            else:
                return [] # No moves possible

        return best_moves

    def _find_immediate_threats(self, board, player):
        """Finds moves that create a 5-in-a-row for the given player."""
        threat_moves = []
        opponent = 3 - player
        directions = [(1, 0), (0, 1), (1, 1), (1, -1)]

        for r in range(board.size):
            for c in range(board.size):
                if board.grid[r, c] == 0: # Consider placing a stone here
                    # Temporarily place stone to check if it creates 5-in-a-row
                    board.grid[r, c] = player
                    is_threat = False
                    for dr, dc in directions:
                        count = 1
                        # Count forward
                        for i in range(1, 6):
                            nr, nc = r + i * dr, c + i * dc
                            if 0 <= nr < board.size and 0 <= nc < board.size and board.grid[nr, nc] == player:
                                count += 1
                            else:
                                break
                        # Count backward
                        for i in range(1, 6):
                            nr, nc = r - i * dr, c - i * dc
                            if 0 <= nr < board.size and 0 <= nc < board.size and board.grid[nr, nc] == player:
                                count += 1
                            else:
                                break
                        if count >= 5: # Check if placing the stone created a line of 5 (potential win next turn)
                           # Check if this line could become 6 (i.e., is not fully blocked)
                           open_ends = 0
                           # Check forward end
                           nr_f, nc_f = r + (count - (sum(1 for i in range(1, 6) if 0 <= r + i * dr < board.size and 0 <= c + i * dc < board.size and board.grid[r + i * dr, c + i * dc] == player))) * dr, c + (count - (sum(1 for i in range(1, 6) if 0 <= r + i * dr < board.size and 0 <= c + i * dc < board.size and board.grid[r + i * dr, c + i * dc] == player))) * dc
                           nr_f_next, nc_f_next = nr_f + dr, nc_f + dc
                           if 0 <= nr_f_next < board.size and 0 <= nc_f_next < board.size and board.grid[nr_f_next, nc_f_next] == 0:
                               open_ends += 1
                           # Check backward end
                           nr_b, nc_b = r - (count - (sum(1 for i in range(1, 6) if 0 <= r - i * dr < board.size and 0 <= c - i * dc < board.size and board.grid[r - i * dr, c - i * dc] == player))) * dr, c - (count - (sum(1 for i in range(1, 6) if 0 <= r - i * dr < board.size and 0 <= c - i * dc < board.size and board.grid[r - i * dr, c - i * dc] == player))) * dc
                           nr_b_prev, nc_b_prev = nr_b - dr, nc_b - dc
                           if 0 <= nr_b_prev < board.size and 0 <= nc_b_prev < board.size and board.grid[nr_b_prev, nc_b_prev] == 0:
                               open_ends += 1

                           if open_ends > 0:
                               is_threat = True
                               break # Found a threat for this move

                    board.grid[r, c] = 0 # Revert the temporary placement
                    if is_threat:
                        threat_moves.append((r, c))
        return threat_moves

    def _get_remaining_moves(self, board, first_move):
        
        temp_board = copy.deepcopy(board)
        temp_board.place_stone(first_move[0], first_move[1])
        return temp_board.get_available_moves()

    def _minimax(self, board, depth, is_maximizing, alpha, beta):
        self.nodes_evaluated += 1

        # if time limit is reached
        if time.time() - self.start_time > self.max_time * 0.9:
            return self._evaluate_board(board)

        # transposition table
        board_hash = board.get_board_state_hash()
        if board_hash in self.transposition_table:
            stored_depth, stored_score = self.transposition_table[board_hash]
            if stored_depth >= depth:
                return stored_score

        if board.game_over or depth == 0:
            return self._evaluate_board(board)

        stones_to_place = 1 if board.first_move else 2

        if is_maximizing:
            max_score = float("-inf")

            
            available_moves = self._get_smart_moves(board)

            
            if stones_to_place == 2:
                
                max_moves_to_consider = min(8, len(available_moves))
                considered_moves = available_moves[:max_moves_to_consider]

               
                for i in range(len(considered_moves)):
                    for j in range(i + 1, len(considered_moves)):
                        move1 = considered_moves[i]
                        move2 = considered_moves[j]

                        # Make the moves
                        board_copy = copy.deepcopy(board)
                        board_copy.place_stone(move1[0], move1[1])

                        # Make sure if first move wins
                        if board_copy.game_over and board_copy.winner == self.player_number:
                            return 10000  # Ai wins

                        board_copy.place_stone(move2[0], move2[1])

                        # Make sure if second move wins
                        if board_copy.game_over and board_copy.winner == self.player_number:
                            return 10000  # Ai wins

                        board_copy.next_turn()

                        score = self._minimax(board_copy, depth - 1, False, alpha, beta)
                        max_score = max(max_score, score)

                        alpha = max(alpha, max_score)
                        if beta <= alpha:
                            break

                    if beta <= alpha:
                        break
            else:
                # For placing one stone
                for move in available_moves:
                    row, col = move

                    
                    board_copy = copy.deepcopy(board)
                    board_copy.place_stone(row, col)

                    if board_copy.game_over and board_copy.winner == self.player_number:
                        return 10000  # Ai wins

                    # Complete the turn
                    board_copy.next_turn()

                    score = self._minimax(board_copy, depth - 1, False, alpha, beta)
                    max_score = max(max_score, score)

                    # Alpha-Beta pruning
                    alpha = max(alpha, max_score)
                    if beta <= alpha:
                        break

            self.transposition_table[board_hash] = (depth, max_score)
            return max_score
        else:
            min_score = float("inf")

            
            available_moves = self._get_smart_moves(board)

           
            if stones_to_place == 2:
                
                max_moves_to_consider = min(8, len(available_moves))
                considered_moves = available_moves[:max_moves_to_consider]

               
                for i in range(len(considered_moves)):
                    for j in range(i + 1, len(considered_moves)):
                        move1 = considered_moves[i]
                        move2 = considered_moves[j]

                        
                        board_copy = copy.deepcopy(board)
                        board_copy.place_stone(move1[0], move1[1])

                        if board_copy.game_over and board_copy.winner == self.opponent_number:
                            return -10000  # 5asm wins

                        board_copy.place_stone(move2[0], move2[1])

                        # Make sure if second move wins
                        if board_copy.game_over and board_copy.winner == self.opponent_number:
                            return -10000  # 5asm wins

                        board_copy.next_turn()

                        # Recursive call
                        score = self._minimax(board_copy, depth - 1, True, alpha, beta)
                        min_score = min(min_score, score)

                        beta = min(beta, min_score)
                        if beta <= alpha:
                            break

                    if beta <= alpha:
                        break
            else:
                # For placing one stone
                for move in available_moves:
                    row, col = move

                    
                    board_copy = copy.deepcopy(board)
                    board_copy.place_stone(row, col)

                    if board_copy.game_over and board_copy.winner == self.opponent_number:
                        return -10000

                    # Complete the turn
                    board_copy.next_turn()

                    score = self._minimax(board_copy, depth - 1, True, alpha, beta)
                    min_score = min(min_score, score)

                    beta = min(beta, min_score)
                    if beta <= alpha:
                        break

            self.transposition_table[board_hash] = (depth, min_score)
            return min_score

    def _evaluate_board(self, board):
        if self.heuristic_type == "advanced":
            return Connect6Heuristics.advanced_weighted_heuristic(board, self.player_number)
        else:
            return Connect6Heuristics.basic_pattern_heuristic(board, self.player_number)

    def _get_smart_moves(self, board):
       
        total_stones = np.count_nonzero(board.grid)
        if total_stones < 5:
            center = board.size // 2
            center_region = min(3, board.size // 6)
            center_moves = []

            for r in range(center - center_region, center + center_region + 1):
                for c in range(center - center_region, center + center_region + 1):
                    if 0 <= r < board.size and 0 <= c < board.size and board.grid[r, c] == 0:
                        distance = max(abs(r - center), abs(c - center))
                        center_moves.append((distance, (r, c)))

            center_moves.sort()
            return [move for _, move in center_moves]

        
        all_moves = board.get_available_moves()
        if not all_moves:
            return []

        # Calculate scores for each move
        move_scores = []
        for move in all_moves:
            r, c = move
            score = self._calculate_move_score(board, r, c)
            move_scores.append((score, move))

        move_scores.sort(reverse=True)

        # Return the top moves
        max_moves = min(30, len(move_scores))
        return [move for _, move in move_scores[:max_moves]]

    def _calculate_move_score(self, board, row, col):
        score = 0

        for dr in range(-2, 3):
            for dc in range(-2, 3):
                r, c = row + dr, col + dc
                if 0 <= r < board.size and 0 <= c < board.size:
                    if board.grid[r, c] != 0:
                        # Closer stones have higher value
                        distance = max(abs(dr), abs(dc))
                        if distance == 1:
                            score += 5
                        else:
                            score += 2

       
        board_copy = copy.deepcopy(board)

        board_copy.grid[row, col] = self.player_number
        ai_score = self._quick_evaluate_position(board_copy, row, col, self.player_number)
        score += ai_score * 2

        board_copy.grid[row, col] = self.opponent_number
        opponent_score = self._quick_evaluate_position(board_copy, row, col, self.opponent_number)
        score += opponent_score * 2.5  # prioritize blocking opponent

        board_copy.grid[row, col] = 0

        return score

    def _quick_evaluate_position(self, board, row, col, player_number):
        score = 0
        directions = [
            (1, 0),
            (0, 1),
            (1, 1),   
            (1, -1)   
        ]

        for dr, dc in directions:
            # Count consecutive stones
            consecutive = 1
            empty_before = 0
            empty_after = 0

            r, c = row + dr, col + dc
            while 0 <= r < board.size and 0 <= c < board.size:
                if board.grid[r, c] == player_number:
                    consecutive += 1
                elif board.grid[r, c] == 0:
                    empty_after += 1
                    break
                else:
                    break
                r += dr
                c += dc

            r, c = row - dr, col - dc
            while 0 <= r < board.size and 0 <= c < board.size:
                if board.grid[r, c] == player_number:
                    consecutive += 1
                elif board.grid[r, c] == 0:
                    empty_before += 1
                    break
                else:
                    break
                r -= dr
                c -= dc

            # Results based on pattern
            if consecutive >= 6:
                score += 1000  #  Win

            elif consecutive == 5 and (empty_before > 0 or empty_after > 0):
                score += 500  # One move away from win
            elif consecutive == 4 and empty_before > 0 and empty_after > 0:
                score += 200
            elif consecutive == 3 and empty_before > 0 and empty_after > 0:
                score += 50   # Medium threat
            elif consecutive == 2 and empty_before > 0 and empty_after > 0:
                score += 10   # Weak threat

        return score

class Button:
    def __init__(self, x, y, width, height, text, color, hover_color, text_color=BLACK, font=font):
        self.rect = pygame.Rect(x, y, width, height)
        self.text = text
        self.color = color
        self.hover_color = hover_color
        self.text_color = text_color
        self.font = font
        self.is_hovered = False

    def draw(self, surface):
        color = self.hover_color if self.is_hovered else self.color
        pygame.draw.rect(surface, color, self.rect)
        pygame.draw.rect(surface, BLACK, self.rect, 2)  # Border

        text_surface = self.font.render(self.text, True, self.text_color)
        text_rect = text_surface.get_rect(center=self.rect.center)
        surface.blit(text_surface, text_rect)

    def update(self, mouse_pos):
        self.is_hovered = self.rect.collidepoint(mouse_pos)

    def is_clicked(self, mouse_pos, mouse_clicked):
        return self.rect.collidepoint(mouse_pos) and mouse_clicked


class RadioButton:
    def __init__(self, x, y, radius, text, color, selected_color, text_color=BLACK, font=small_font):
        self.x = x
        self.y = y
        self.radius = radius
        self.text = text
        self.color = color
        self.selected_color = selected_color
        self.text_color = text_color
        self.font = font
        self.is_selected = False

    def draw(self, surface):
        pygame.draw.circle(surface, BLACK, (self.x, self.y), self.radius)

        if self.is_selected:
            pygame.draw.circle(surface, self.selected_color, (self.x, self.y), self.radius - 2)
        else:
            pygame.draw.circle(surface, WHITE, (self.x, self.y), self.radius - 2)

        text_surface = self.font.render(self.text, True, self.text_color)
        text_rect = text_surface.get_rect(midleft=(self.x + self.radius + 5, self.y))
        surface.blit(text_surface, text_rect)

    def is_clicked(self, mouse_pos):
        x, y = mouse_pos
        distance = math.sqrt((x - self.x) ** 2 + (y - self.y) ** 2)
        return distance <= self.radius


class RadioGroup:
    def __init__(self, buttons):
        self.buttons = buttons
        if buttons:
            buttons[0].is_selected = True

    def update(self, mouse_pos, mouse_clicked):
        if mouse_clicked:
            for button in self.buttons:
                if button.is_clicked(mouse_pos):
                    self.select_button(button)
                    break

    def select_button(self, selected_button):
        for button in self.buttons:
            button.is_selected = (button == selected_button)

    def draw(self, surface):
        for button in self.buttons:
            button.draw(surface)

    def get_selected_text(self):
        for button in self.buttons:
            if button.is_selected:
                return button.text
        return None


def draw_board(board):
    
    board_width = CELL_SIZE * board.size
    board_height = CELL_SIZE * board.size

    max_board_width = CELL_SIZE * MAX_BOARD_SIZE
    max_board_height = CELL_SIZE * MAX_BOARD_SIZE

    
    
    board_x = HORIZONTAL_PADDING + (WINDOW_WIDTH - board_width) // 2 
    # Constant Board Center
    max_board_y = MARGIN - VERTICAL_PADDING
    board_y = max_board_y + (max_board_height - board_height) // 2


    board_rect = pygame.Rect(board_x, board_y, board_width, board_height)
    pygame.draw.rect(screen, BOARD_COLOR, board_rect)

    # grid lines
    for i in range(board.size):
        
        pygame.draw.line(
            screen, GRID_COLOR,
            (board_x, board_y + i * CELL_SIZE),
            (board_x + (board.size - 1) * CELL_SIZE, board_y + i * CELL_SIZE),
            2 if i == 0 or i == board.size - 1 else 1
        )

        
        pygame.draw.line(
            screen, GRID_COLOR,
            (board_x + i * CELL_SIZE, board_y),
            (board_x + i * CELL_SIZE, board_y + (board.size - 1) * CELL_SIZE),
            2 if i == 0 or i == board.size - 1 else 1
        )

    if board.size >= 13:
        if board.size == 19:
            star_points = [(3, 3), (3, 9), (3, 15), (9, 3), (9, 9), (9, 15), (15, 3), (15, 9), (15, 15)]
        elif board.size == 13:
            star_points = [(3, 3), (3, 9), (6, 6), (9, 3), (9, 9)]
        else:
            margin = min(3, board.size // 6)
            mid = board.size // 2
            star_points = [
                (margin, margin), (margin, board.size - margin - 1),
                (board.size - margin - 1, margin), (board.size - margin - 1, board.size - margin - 1),
                (mid, mid)
            ]

        for r, c in star_points:
            pygame.draw.circle(
                screen, BLACK,
                (board_x + c * CELL_SIZE, board_y + r * CELL_SIZE),
                3  # kanet 4
            )

    for r in range(board.size):
        for c in range(board.size):
            if board.grid[r, c] != 0:
                # Determine stone color
                color = BLACK if board.grid[r, c] == 1 else WHITE

            
                pygame.draw.circle(
                    screen, color,
                    (board_x + c * CELL_SIZE, board_y + r * CELL_SIZE),
                    STONE_RADIUS
                )

                #  border for white stones
                if board.grid[r, c] == 2:
                    pygame.draw.circle(
                        screen, BLACK,
                        (board_x + c * CELL_SIZE, board_y + r * CELL_SIZE),
                        STONE_RADIUS, 1
                    )

    if board.last_move:
        r, c = board.last_move
        pygame.draw.circle(
            screen, RED,
            (board_x + c * CELL_SIZE, board_y + r * CELL_SIZE),
            STONE_RADIUS // 2, 2
        )

    # Highlight winning stones
    if board.winning_stones:
        for r, c in board.winning_stones:
         pygame.draw.circle(
                screen, GOLD,
                (board_x + c * CELL_SIZE, board_y + r * CELL_SIZE),
                STONE_RADIUS + 2, 2
            )

    
    return board_x, board_y, board_width, board_height


def draw_info_panel(board, ai_player=None, thinking=False):
    max_board_width = CELL_SIZE * MAX_BOARD_SIZE
    max_board_height = CELL_SIZE * MAX_BOARD_SIZE

    
    info_x = HORIZONTAL_PADDING + (WINDOW_WIDTH - max_board_width) // 2 
    info_y = MARGIN + max_board_height - VERTICAL_PADDING

    #  info panel background
    info_rect = pygame.Rect(
        info_x,
        info_y,
        max_board_width,
        INFO_HEIGHT
    )
    pygame.draw.rect(screen, LIGHT_BLUE, info_rect)
    pygame.draw.rect(screen, BLACK, info_rect, 2)

    state_text = board.get_state_string()
    if thinking:
        state_text += " (AI thinking...)"

    state_surface = large_font.render(state_text, True, BLACK)
    state_rect = state_surface.get_rect(
        midtop=(info_rect.centerx, info_rect.top + 10)
    )
    screen.blit(state_surface, state_rect)

    turn_text = f"Turn: {board.turn_count}"
    turn_surface = font.render(turn_text, True, BLACK)
    turn_rect = turn_surface.get_rect(
        topleft=(info_rect.left + 10, state_rect.bottom + 10)
    )
    screen.blit(turn_surface, turn_rect)

    #  board size
    size_text = f"Board Size: {board.size}×{board.size}"
    size_surface = font.render(size_text, True, BLACK)
    size_rect = size_surface.get_rect(
        topright=(info_rect.right - 10, turn_rect.top)
    )
    screen.blit(size_surface, size_rect)

    if ai_player:
        ai_color = "Black" if ai_player.player_number == 1 else "White"
        ai_text = f"AI: {ai_color} (Heuristic: {ai_player.heuristic_type})"
        ai_surface = font.render(ai_text, True, BLACK)
        ai_rect = ai_surface.get_rect(
            topleft=(turn_rect.left, turn_rect.bottom + 5)
        )
        screen.blit(ai_surface, ai_rect)

        if hasattr(ai_player, 'nodes_evaluated') and hasattr(ai_player, 'start_time'):
            elapsed = time.time() - ai_player.start_time if ai_player.start_time > 0 else 0
            stats_text = f"Nodes evaluated: {ai_player.nodes_evaluated}, Time: {elapsed:.2f}s"
            stats_surface = font.render(stats_text, True, BLACK)
            stats_rect = stats_surface.get_rect(
                topleft=(ai_rect.left, ai_rect.bottom + 5)
            )
            screen.blit(stats_surface, stats_rect)

   

    return info_rect


def draw_buttons(info_rect):
    
    max_board_width = CELL_SIZE * MAX_BOARD_SIZE

    
    # button_x = (WINDOW_WIDTH - max_board_width) // 2 # Original centering relative to WINDOW_WIDTH
    button_x = HORIZONTAL_PADDING + (WINDOW_WIDTH - max_board_width) // 2 # Centering within the padded area
    button_y = info_rect.bottom + BUTTON_MARGIN
    button_width = (max_board_width - BUTTON_MARGIN * 3) // 4

    new_game_button = Button(
        button_x, button_y, button_width, BUTTON_HEIGHT,
        "New Game", GREEN, LIGHT_GREEN
    )

    undo_button = Button(
        button_x + button_width + BUTTON_MARGIN, button_y, button_width, BUTTON_HEIGHT,
        "Undo", BLUE, LIGHT_BLUE
    )

    hint_button = Button(
        button_x + (button_width + BUTTON_MARGIN) * 2, button_y, button_width, BUTTON_HEIGHT,
        "Hint", GOLD, LIGHT_BLUE
    )

    quit_button = Button(
        button_x + (button_width + BUTTON_MARGIN) * 3, button_y, button_width, BUTTON_HEIGHT,
        "Quit", RED, LIGHT_BLUE
    )

    #  buttons
    new_game_button.draw(screen)
    undo_button.draw(screen)
    hint_button.draw(screen)
    quit_button.draw(screen)

    return new_game_button, undo_button, hint_button, quit_button


def draw_settings_panel(info_rect, button_y, player_type_group, heuristic_group, board_size_group):
    max_board_width = CELL_SIZE * MAX_BOARD_SIZE

    # Makan settings panel at a fixed position below the buttons
   
    settings_x = HORIZONTAL_PADDING + (WINDOW_WIDTH - max_board_width) // 2 
    settings_y = button_y + BUTTON_HEIGHT + BUTTON_MARGIN

    #  settings panel background
    settings_rect = pygame.Rect(
        settings_x,
        settings_y,
        max_board_width,
        SETTINGS_HEIGHT
    )
    pygame.draw.rect(screen, LIGHT_GREEN, settings_rect)
    pygame.draw.rect(screen, BLACK, settings_rect, 2)

    #  settings title
    title_surface = large_font.render("Game Settings", True, BLACK)
    title_rect = title_surface.get_rect(
        midtop=(settings_rect.centerx, settings_rect.top + 5)
    )
    screen.blit(title_surface, title_rect)

    col1_x = settings_rect.left + 20
    col2_x = settings_rect.centerx - 50
    col3_x = settings_rect.right - 150

    #  player type label
    player_label = font.render("Player Type:", True, BLACK)
    screen.blit(player_label, (col1_x, title_rect.bottom + 10))

    #  heuristic label
    heuristic_label = font.render("AI Heuristic:", True, BLACK)
    screen.blit(heuristic_label, (col2_x, title_rect.bottom + 10))

    #  board size label
    board_size_label = font.render("Board Size:", True, BLACK)
    screen.blit(board_size_label, (col3_x, title_rect.bottom + 10))

    #  radio groups
    for i, button in enumerate(player_type_group.buttons):
        button.x = col1_x + 10
        button.y = title_rect.bottom + 35 + i * 20

    for i, button in enumerate(heuristic_group.buttons):
        button.x = col2_x + 10
        button.y = title_rect.bottom + 35 + i * 20

    for i, button in enumerate(board_size_group.buttons):
        button.x = col3_x + 10
        button.y = title_rect.bottom + 35 + i * 20

    player_type_group.draw(screen)
    heuristic_group.draw(screen)
    board_size_group.draw(screen)

    # Return the settings panel position
    return settings_rect


def get_board_position(mouse_pos, board_position):
    board_x, board_y, board_width, board_height = board_position
    board_size = board_width // CELL_SIZE

    x, y = mouse_pos

    if (x < board_x or x >= board_x + board_width or
        y < board_y or y >= board_y + board_height):
        return None

   
    col = round((x - board_x) / CELL_SIZE)
    row = round((y - board_y) / CELL_SIZE)

    
    if not (0 <= row < board_size and 0 <= col < board_size):
        return None

    return row, col


def main():
  
    board_size = DEFAULT_BOARD_SIZE

    # Create game board
    board = Connect6Board(board_size)

    # Create Ai player 
    ai_player = None
    ai_thinking = False
    ai_move = None

    player_type_buttons = [
        RadioButton(0, 0, 8, "Human vs Human", GREEN, GREEN),
        RadioButton(0, 0, 8, "Human vs AI (Black)", BLUE, BLUE),
        RadioButton(0, 0, 8, "Human vs AI (White)", BLUE, BLUE)
    ]
    player_type_group = RadioGroup(player_type_buttons)

    # Heuristics buttons
    heuristic_buttons = [
        RadioButton(0, 0, 8, "Basic", GREEN, GREEN),
        RadioButton(0, 0, 8, "Advanced", BLUE, BLUE)
    ]
    heuristic_group = RadioGroup(heuristic_buttons)

    # Grid Sizes buttons
    board_size_buttons = [
        RadioButton(0, 0, 8, "19×19", GREEN, GREEN),
        RadioButton(0, 0, 8, "15×15", BLUE, BLUE),
        RadioButton(0, 0, 8, "13×13", BLUE, BLUE),
        RadioButton(0, 0, 8, "9×9", BLUE, BLUE)
    ]
    board_size_group = RadioGroup(board_size_buttons)

    # game state
    running = True
    clock = pygame.time.Clock()
    settings_confirmed = False
    hint_move = None

    while running:
        # events
        mouse_pos = pygame.mouse.get_pos()
        mouse_clicked = False

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            elif event.type == pygame.MOUSEBUTTONDOWN:
                if event.button == 1:
                    mouse_clicked = True

        
        screen.fill(BEIGE)


        # Left Text
        left_text_str = "Connect 6 - The Game"
        left_text_surface = Big_font.render(left_text_str, True, BLACK)
        left_text_rect = left_text_surface.get_rect()
        
        left_text_rect.centerx = HORIZONTAL_PADDING // 2
        left_text_rect.centery = screen.get_height() // 2 
        
        if HORIZONTAL_PADDING > 20: 
             if left_text_rect.right >= HORIZONTAL_PADDING: 
                  left_text_rect.right = HORIZONTAL_PADDING - 10 
             if left_text_rect.left < 10:
                  left_text_rect.left = 10 
             screen.blit(left_text_surface, left_text_rect)
                    # Game Rules Text
        rules_font = pygame.font.SysFont("Arial", 26)  # smaller font size
        rules_lines = [
            "Game Rules:",
            "Black plays first, putting one black stone",
            "on one intersection.",
            "Subsequently, White and Black take turns,",
            "placing two stones on two different",
            "unoccupied spaces each turn.",
            "Winner: The player who is the first to get",
            "six or more stones in a row (horizontally,",
            "vertically, or diagonally) wins."
        ]

        # Render and display each line
        line_spacing = 5
        current_y = left_text_rect.bottom + 10  # start just below the title

        for line in rules_lines:
            rule_surface = rules_font.render(line, True, BLACK)
            rule_rect = rule_surface.get_rect()
            rule_rect.left = left_text_rect.left  # align with title
            rule_rect.top = current_y
            screen.blit(rule_surface, rule_rect)
            current_y += rule_rect.height + line_spacing


       

        
        board_position = draw_board(board)

        
        info_rect = draw_info_panel(board, ai_player, ai_thinking)

        new_game_button, undo_button, hint_button, quit_button = draw_buttons(info_rect)

        
        new_game_button.update(mouse_pos)
        undo_button.update(mouse_pos)
        hint_button.update(mouse_pos)
        quit_button.update(mouse_pos)

        button_y = info_rect.bottom + BUTTON_MARGIN
        settings_rect = draw_settings_panel(info_rect, button_y, player_type_group, heuristic_group, board_size_group)

        #  hint move 
        if settings_confirmed and hint_move and not board.game_over:
            hint_row, hint_col = hint_move
            hint_rect = pygame.Rect(
                board_position[0] + hint_col * CELL_SIZE - STONE_RADIUS,
                board_position[1] + hint_row * CELL_SIZE - STONE_RADIUS,
                STONE_RADIUS * 2, STONE_RADIUS * 2
            )
            pygame.draw.rect(screen, GOLD, hint_rect, 2)

        
        if not settings_confirmed:
            # Overlay el warah el setting message
            overlay = pygame.Surface((board_position[2], board_position[3]), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 128))  # Semi-transparent black
            screen.blit(overlay, (board_position[0], board_position[1]))

            # Message
            message_surface = large_font.render("Select Settings and Press New Game to Start", True, WHITE)
            message_rect = message_surface.get_rect(center=(board_position[0] + board_position[2] // 2, board_position[1] + board_position[3] // 2))
            screen.blit(message_surface, message_rect)

        if mouse_clicked:
            if new_game_button.is_clicked(mouse_pos, mouse_clicked):
                selected_size_text = board_size_group.get_selected_text()
                new_board_size = int(selected_size_text.split('×')[0])

                board_size = new_board_size
                board = Connect6Board(board_size)
                hint_move = None

                # Create Ai player based on settings
                player_type = player_type_group.get_selected_text()
                heuristic = heuristic_group.get_selected_text().lower()

                if player_type == "Human vs AI (Black)":
                    ai_player = Connect6AI(1, heuristic)
                elif player_type == "Human vs AI (White)":
                    ai_player = Connect6AI(2, heuristic)
                else:
                    ai_player = None

                settings_confirmed = True

            elif undo_button.is_clicked(mouse_pos, mouse_clicked) and settings_confirmed:
                if board.undo_last_move():
                    hint_move = None

                    
                    if ai_player and ai_player.player_number != board.current_player:
                        board.undo_last_move()

            elif hint_button.is_clicked(mouse_pos, mouse_clicked) and settings_confirmed:
                if not board.game_over and not ai_thinking:
                    hint_ai = Connect6AI(board.current_player, 'basic')
                    hint_move = hint_ai.get_move(board)

            elif quit_button.is_clicked(mouse_pos, mouse_clicked):
                running = False

            player_type_group.update(mouse_pos, mouse_clicked)
            heuristic_group.update(mouse_pos, mouse_clicked)
            board_size_group.update(mouse_pos, mouse_clicked)

            if settings_confirmed and not board.game_over and not ai_thinking:
                is_ai_turn = (ai_player and ai_player.player_number == board.current_player)

                if not is_ai_turn:
                    board_pos = get_board_position(mouse_pos, board_position)
                    if board_pos:
                        row, col = board_pos
                        if board.place_stone(row, col):
                            hint_move = None

        if settings_confirmed and not board.game_over and ai_player and ai_player.player_number == board.current_player and not ai_thinking:
            ai_thinking = True
            ai_move = None

            def calculate_ai_move():
                nonlocal ai_move
                ai_move = ai_player.get_move(board)

            import threading
            ai_thread = threading.Thread(target=calculate_ai_move)
            ai_thread.daemon = True
            ai_thread.start()

        if ai_thinking and ai_move is not None:
            # Make the Ai move
            row, col = ai_move
            if board.place_stone(row, col):

                if not board.first_move and board.stones_placed_this_turn < 2 and not board.game_over and board.current_player == ai_player.player_number:
                    second_move = ai_player.get_second_move(board)
                    row2, col2 = second_move
                    if board.place_stone(row2, col2):
                        pass

            ai_thinking = False
            ai_move = None

        pygame.display.flip()
        clock.tick(60)

    pygame.quit()

# run the game
if __name__ == "__main__":
    main()